# A reverse declaration is outside the union

Which declaration is wrong, and how do we repair it?

Run cells from top to bottom in a fresh Python kernel using the repository environment. Set the working directory to the repository root. This notebook is self-contained; do not run another declaration experiment in the same kernel. The matching `.py` file runs with `uv run python examples/step_21_relations/35_error_outside_alternative.py`.

This is a deliberately broken declaration. The error is caught and printed; the corrected declaration is explained at the end. Read the [diagnostic guide](../../docs/how-to/specialization-declaration-fails.md) for the working baseline.


## Imports

These imports supply the annotation tools and AOA types used below.

In [1]:
from __future__ import annotations

from typing import Annotated, Literal

from pydantic import Field

from aoa.action_machine.domain import BaseEntity, Classifier, Generalization, Inverse, Rel, Specialization
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.intents.entity import entity


## MusicDomain

Group the music catalogue declarations. The field descriptions explain the data each object carries. References to classes defined later are resolved after all declarations are present.

In [2]:
class MusicDomain(BaseDomain):
    """Group the music catalogue declarations."""

    name = "music"
    description = "A music catalogue"


## VinylRecordEntity

Describe the information shared by all pressings. The field descriptions explain the data each object carries. References to classes defined later are resolved after all declarations are present.

In [3]:
@entity(description="Vinyl record", domain=MusicDomain)
class VinylRecordEntity(BaseEntity):
    """Describe the information shared by all pressings."""

    id: str = Field(description="Record identifier")
    title: str = Field(description="Album title")
    media: str = Field(description="Pressing code")
    pressing: Annotated[
        Specialization[FirstPressEntity | RepressEntity | TestPressEntity],
        Classifier(field="media", codes=Literal["first", "repress", "test"]),
        Inverse(field_name="record"),
    ] = Rel(description="Details of this pressing")


## FirstPressEntity

Describe the stamper used for a first pressing. The field descriptions explain the data each object carries. References to classes defined later are resolved after all declarations are present.

In [4]:
@entity(description="First pressing", domain=MusicDomain)
class FirstPressEntity(BaseEntity):
    """Describe the stamper used for a first pressing."""

    id: str = Field(description="Pressing identifier")
    stamper: str = Field(description="Stamper code")
    record: Annotated[
        Generalization[VinylRecordEntity],
        Classifier("record", Literal["first"]),
        Inverse(VinylRecordEntity, "pressing"),
    ] = Rel(description="Record described by this pressing")


## RepressEntity

Describe when an album was pressed again. The field descriptions explain the data each object carries. References to classes defined later are resolved after all declarations are present.

In [5]:
@entity(description="Re-pressing", domain=MusicDomain)
class RepressEntity(BaseEntity):
    """Describe when an album was pressed again."""

    id: str = Field(description="Pressing identifier")
    year: int = Field(description="Re-pressing year")
    record: Annotated[
        Generalization[VinylRecordEntity],
        Classifier("record", Literal["repress"]),
        Inverse(VinylRecordEntity, "pressing"),
    ] = Rel(description="Record described by this pressing")


## TestPressEntity

Describe who approved a test pressing. The field descriptions explain the data each object carries. References to classes defined later are resolved after all declarations are present.

In [6]:
@entity(description="Test pressing", domain=MusicDomain)
class TestPressEntity(BaseEntity):
    """Describe who approved a test pressing."""

    id: str = Field(description="Pressing identifier")
    approved_by: str = Field(description="Reviewer name")
    record: Annotated[
        Generalization[VinylRecordEntity],
        Classifier("record", Literal["test"]),
        Inverse(VinylRecordEntity, "pressing"),
    ] = Rel(description="Record described by this pressing")


## UnlistedPressEntity

Demonstrate an alternative omitted from the head. The field descriptions explain the data each object carries. References to classes defined later are resolved after all declarations are present.

In [7]:
@entity(description="An unlisted pressing", domain=MusicDomain)
class UnlistedPressEntity(BaseEntity):
    """Demonstrate an alternative omitted from the head."""

    id: str = Field(description="Pressing identifier")
    record: Annotated[
        Generalization[VinylRecordEntity],
        Classifier("record", Literal["unlisted"]),
        Inverse(VinylRecordEntity, "pressing"),
    ] = Rel(description="Record described by this pressing")


## Resolve the declared types

Pydantic can now resolve the references between classes. This does not create record objects or load data.

In [8]:
UnlistedPressEntity.model_rebuild()


## Resolve the declared types

Pydantic can now resolve the references between classes. This does not create record objects or load data.

In [9]:
for model in (VinylRecordEntity, FirstPressEntity, RepressEntity, TestPressEntity):
    model.model_rebuild()


## The experiment

Which declaration is wrong, and how do we repair it?

In [10]:
from aoa.action_machine.domain.exceptions import SpecializationDeclarationError
from aoa.action_machine.graph.validators.entity_specialization_validator import validate_entity_specializations

try:
    validate_entity_specializations()
except SpecializationDeclarationError as error:
    print(type(error).__name__ + ": " + str(error))
else:
    raise AssertionError("The broken declaration was accepted")


SpecializationDeclarationError: Specialization 'pressing' on entity 'VinylRecordEntity' is invalid: class 'UnlistedPressEntity' points at 'pressing' without being named in the union — add it to the alternatives or remove its reverse field


## Why this result occurs

The loaded UnlistedPressEntity declaration points at VinylRecordEntity but is absent from its alternatives. The validator discovers loaded declared classes, not just classes reachable from the head union.

## Repair

Remove this unintended class. To make it a supported alternative instead, add both its class and unlisted code to the head.

Apply that change to the declaration above, restart the kernel, and run again. Replace the experiment cell with the following success check. This corrected model has also been run through an actual machine build:

```python
from aoa.action_machine.graph.validators.entity_specialization_validator import validate_entity_specializations
from aoa.action_machine.runtime.action_product_machine import ActionProductMachine

validate_entity_specializations()
ActionProductMachine(loggers=[])
print("Corrected model built")
```

Expected output:

```text
Corrected model built
```